# Chart Prototypes: Housing Affordability in Denmark, 1995–2024

Prototype notebook — one chart per section. Change the **parameters cell** below
to switch year, property type, family type, or income basis; all charts update.

**Research questions:**
1. How have house prices and income evolved in Denmark since 1995?
   - 1a. What is the trend of house prices in the different provinces?
   - 1b. How has affordability changed for couples compared to singles?
2. Is housing more affordable east or west of the Great Belt?
3. Is it worth living in Copenhagen? Is the affordability gap growing?

## Parameters
Change these and re-run the notebook to switch views.

In [ ]:
YEAR = 2024
PROPERTY_TYPE = "one_family_house"      # "one_family_house" or "owner_occupied_flat"
FAMILY_TYPE   = "all_families"          # "all_families", "couples", or "singles"
INCOME_BASIS  = "disposable"            # "disposable" or "pretax"

## Setup

In [ ]:
from pathlib import Path
import json, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore", category=FutureWarning)

REPO_ROOT      = Path("..").resolve()
DATA_DIR       = REPO_ROOT / "data" / "processed"
NB_FIGURES     = Path("figures")
REPORT_FIGURES = REPO_ROOT / "report" / "figures"

NB_FIGURES.mkdir(exist_ok=True)
REPORT_FIGURES.mkdir(exist_ok=True)

PNG_WIDTH = 1600
PNG_SCALE = 2

In [ ]:
df = pd.read_csv(DATA_DIR / "affordability_annual.csv")

checks = {
    "Row count is 1,980": len(df) == 1980,
    "No NaN values": df.notna().all().all(),
    "11 landsdele": df["landsdel"].nunique() == 11,
    "30 years (1995-2024)": df["year"].nunique() == 30,
    "ratio_disposable ≈ price / disposable income": np.allclose(
        df["ratio_disposable"],
        df["avg_price_dkk"] / df["avg_disposable_income_dkk"],
        rtol=1e-3,
    ),
    "ratio_pretax ≈ price / pretax income": np.allclose(
        df["ratio_pretax"],
        df["avg_price_dkk"] / df["avg_pretax_income_dkk"],
        rtol=1e-3,
    ),
}

for label, ok in checks.items():
    print(f"  {'✓' if ok else '✗'} {label}")

assert all(checks.values()), "One or more sanity checks failed"

In [ ]:
BORNHOLM_SIDE = "east"  # change to "west" if Bornholm should be grouped with Jutland

EAST_LANDSDELE = [
    "Byen København", "Københavns omegn", "Nordsjælland",
    "Østsjælland", "Vest- og Sydsjælland",
]
WEST_LANDSDELE = [
    "Fyn", "Sydjylland", "Østjylland", "Vestjylland", "Nordjylland",
]

if BORNHOLM_SIDE == "east":
    EAST_LANDSDELE = EAST_LANDSDELE + ["Bornholm"]
else:
    WEST_LANDSDELE = WEST_LANDSDELE + ["Bornholm"]

_side_map = {ld: "east" for ld in EAST_LANDSDELE} | {ld: "west" for ld in WEST_LANDSDELE}
df["side"] = df["landsdel"].map(_side_map)
assert df["side"].notna().all(), "Some landsdele were not assigned to east/west"
print(f"East ({len(EAST_LANDSDELE)}): {', '.join(EAST_LANDSDELE)}")
print(f"West ({len(WEST_LANDSDELE)}): {', '.join(WEST_LANDSDELE)}")

In [ ]:
def get_columns(basis):
    """Return (ratio_col, income_col) for the chosen income basis."""
    if basis == "disposable":
        return "ratio_disposable", "avg_disposable_income_dkk"
    return "ratio_pretax", "avg_pretax_income_dkk"

RATIO_COL, INCOME_COL = get_columns(INCOME_BASIS)


def weighted_aggregate(grp):
    """Weighted price (by n_sales) and income (by n_families), then ratio."""
    w_price  = np.average(grp["avg_price_dkk"], weights=grp["n_sales"])
    w_income = np.average(grp[INCOME_COL],      weights=grp["n_families"])
    return pd.Series({
        "avg_price_dkk": w_price,
        INCOME_COL:      w_income,
        RATIO_COL:       w_price / w_income,
        "n_sales":       grp["n_sales"].sum(),
        "n_families":    grp["n_families"].sum(),
    })


def low_sample_note(n):
    return f"{n:,.0f} ⚠ low sample" if n < 15 else f"{n:,.0f}"

In [ ]:
# Consistent landsdel colours across all charts.
# 8 slots from the colourblind-safe dataviz palette + 3 extended.
# NOTE: 11 categories exceeds the 8-slot CVD-safe limit. For the final
# dashboard, charts showing all 11 should use small multiples or emphasis.
LANDSDEL_COLORS = {
    "Byen København":        "#2a78d6",  # blue
    "Københavns omegn":      "#4a3aa7",  # violet
    "Nordsjælland":          "#1baf7a",  # aqua
    "Østsjælland":           "#e87ba4",  # magenta
    "Vest- og Sydsjælland":  "#eda100",  # yellow
    "Bornholm":              "#7f7f7f",  # gray
    "Fyn":                   "#eb6834",  # orange
    "Sydjylland":            "#008300",  # green
    "Østjylland":            "#e34948",  # red
    "Vestjylland":           "#8c564b",  # brown
    "Nordjylland":           "#17becf",  # teal
}

SIDE_COLORS = {"east": "#2a78d6", "west": "#eb6834"}

# Fixed ordering (east first, then west)
LANDSDEL_ORDER = list(LANDSDEL_COLORS.keys())

# Sequential blue scale for choropleth / heatmap (light → dark)
SEQ_BLUE = [
    [0.00, "#cde2fb"],
    [0.17, "#9ec5f4"],
    [0.33, "#6da7ec"],
    [0.50, "#3987e5"],
    [0.67, "#256abf"],
    [0.83, "#184f95"],
    [1.00, "#0d366b"],
]

In [ ]:
def _topojson_to_geojson(topo, obj_name):
    """Minimal TopoJSON → GeoJSON converter (polygons only)."""
    obj = topo["objects"][obj_name]
    arcs_enc = topo["arcs"]
    tx = topo.get("transform")

    decoded = []
    for arc in arcs_enc:
        pts, x, y = [], 0, 0
        for dx, dy in arc:
            x += dx; y += dy
            if tx:
                pts.append([x * tx["scale"][0] + tx["translate"][0],
                            y * tx["scale"][1] + tx["translate"][1]])
            else:
                pts.append([x, y])
        decoded.append(pts)

    def _resolve(idx):
        return decoded[idx] if idx >= 0 else list(reversed(decoded[~idx]))

    def _ring(indices):
        coords = []
        for i in indices:
            seg = _resolve(i)
            coords.extend(seg if not coords else seg[1:])
        return coords

    features = []
    for g in obj["geometries"]:
        if g["type"] == "Polygon":
            geom = {"type": "Polygon", "coordinates": [_ring(r) for r in g["arcs"]]}
        elif g["type"] == "MultiPolygon":
            geom = {"type": "MultiPolygon",
                    "coordinates": [[_ring(r) for r in poly] for poly in g["arcs"]]}
        else:
            continue
        features.append({"type": "Feature", "properties": g.get("properties", {}), "geometry": geom})
    return {"type": "FeatureCollection", "features": features}


topo_path = DATA_DIR / "landsdele.topo.json"

try:
    import geopandas as gpd
    gdf = gpd.read_file(topo_path)
    geojson = json.loads(gdf.to_json())
    for f in geojson["features"]:
        if "nuts3" not in f["properties"]:
            f["properties"]["nuts3"] = f.get("id", "")
    print("Loaded GeoJSON via geopandas")
except Exception as exc:
    print(f"geopandas failed ({exc}); using manual converter")
    with open(topo_path) as _f:
        _topo = json.load(_f)
    geojson = _topojson_to_geojson(_topo, "landsdele")

print(f"{len(geojson['features'])} features loaded")

In [ ]:
def save_chart(fig, num, short_name, *, png_to_report=True, png_height=900):
    """Save HTML to notebooks/figures/ and PNG to report/figures/."""
    html_path = NB_FIGURES / f"fig{num:02d}_{short_name}.html"
    fig.write_html(str(html_path))
    if png_to_report:
        png_path = REPORT_FIGURES / f"g{num}-{short_name}.png"
        fig.write_image(str(png_path), width=PNG_WIDTH, height=png_height, scale=PNG_SCALE)
    print(f"  Saved fig{num:02d}_{short_name}" + (" (+PNG)" if png_to_report else " (HTML only)"))

## Chart 1 — Choropleth map with year slider

**Research question 1a:** What is the trend of house prices in the different provinces?

**What to look for:** Geographic clustering of high/low ratios; how the colour
gradient shifts over time (pre-2008 boom, post-GFC dip, 2020s surge).

In [ ]:
_choro = df[(df["property_type"] == PROPERTY_TYPE) & (df["family_type"] == FAMILY_TYPE)].copy()
_choro["sales_note"] = _choro["n_sales"].apply(low_sample_note)

_vmin = _choro[RATIO_COL].min()
_vmax = _choro[RATIO_COL].max()

fig1 = px.choropleth(
    _choro.sort_values("year"),
    geojson=geojson,
    locations="nuts3",
    featureidkey="properties.nuts3",
    color=RATIO_COL,
    animation_frame="year",
    hover_name="landsdel",
    hover_data={
        "nuts3": False, "year": False,
        RATIO_COL: ":.2f",
        "sales_note": True,
    },
    color_continuous_scale=SEQ_BLUE,
    range_color=[_vmin, _vmax],
    labels={RATIO_COL: "Price-to-Income Ratio", "sales_note": "Sales"},
)
fig1.update_geos(fitbounds="locations", visible=False)
fig1.update_layout(
    margin=dict(l=0, r=0, t=50, b=0),
    title=f"Affordability ratio by region ({INCOME_BASIS} income)",
    coloraxis_colorbar_title="Ratio",
)
fig1.show()

# Static PNG for the report (YEAR frame only)
_snap = _choro[_choro["year"] == YEAR]
fig1s = px.choropleth(
    _snap, geojson=geojson, locations="nuts3", featureidkey="properties.nuts3",
    color=RATIO_COL, hover_name="landsdel",
    color_continuous_scale=SEQ_BLUE, range_color=[_vmin, _vmax],
    labels={RATIO_COL: "Price-to-Income Ratio"},
)
fig1s.update_geos(fitbounds="locations", visible=False)
fig1s.update_layout(margin=dict(l=0, r=0, t=50, b=0),
                    title=f"Affordability ratio by region, {YEAR}")

save_chart(fig1, 1, "choropleth", png_to_report=False)
save_chart(fig1s, 1, "choropleth")

## Chart 2 — Regional ranking (horizontal bars)

**Research question 2:** Is housing more affordable east or west of the Great Belt?

**What to look for:** Eastern regions (especially Copenhagen) at the top; clean
east/west colour split; how far apart the extremes are.

In [ ]:
_bar = (
    df[(df["year"] == YEAR) & (df["property_type"] == PROPERTY_TYPE) & (df["family_type"] == FAMILY_TYPE)]
    .sort_values(RATIO_COL, ascending=True)
    .copy()
)
_bar["sales_note"] = _bar["n_sales"].apply(low_sample_note)

fig2 = px.bar(
    _bar, x=RATIO_COL, y="landsdel", color="side",
    orientation="h",
    color_discrete_map=SIDE_COLORS,
    hover_data={RATIO_COL: ":.2f", "sales_note": True, "side": False},
    labels={RATIO_COL: "Price-to-Income Ratio", "landsdel": "", "sales_note": "Sales"},
    category_orders={"landsdel": _bar["landsdel"].tolist()},
    title=f"Affordability ranking, {YEAR} — {PROPERTY_TYPE.replace('_',' ').title()}",
)
fig2.update_layout(yaxis_title="", legend_title="Great Belt side")
fig2.show()
save_chart(fig2, 2, "ranking")

## Chart 3 — Affordability ratio over time (11 lines)

**Research questions 1 & 1a:** How have house prices and income evolved? What is
the trend per province?

**What to look for:** Pre-2008 divergence, post-GFC convergence, 2020s re-divergence.
Copenhagen well above the rest. Note: 11 series exceeds the colourblind-safe
maximum of 8; the final dashboard should use emphasis or small multiples.

In [ ]:
_line = df[(df["property_type"] == PROPERTY_TYPE) & (df["family_type"] == FAMILY_TYPE)].copy()
_line["sales_note"] = _line["n_sales"].apply(low_sample_note)

fig3 = px.line(
    _line, x="year", y=RATIO_COL, color="landsdel",
    color_discrete_map=LANDSDEL_COLORS,
    category_orders={"landsdel": LANDSDEL_ORDER},
    hover_data={RATIO_COL: ":.2f", "sales_note": True},
    labels={RATIO_COL: "Price-to-Income Ratio", "year": "Year", "landsdel": "Landsdel",
            "sales_note": "Sales"},
    title=f"Affordability ratio by region, 1995–2024",
)

# Direct labels at the right end
_end = _line[_line["year"] == _line["year"].max()]
for _, r in _end.iterrows():
    fig3.add_annotation(
        x=r["year"], y=r[RATIO_COL],
        text=r["landsdel"], showarrow=False,
        xanchor="left", xshift=6,
        font=dict(size=9, color=LANDSDEL_COLORS.get(r["landsdel"], "#333")),
    )

fig3.update_layout(
    showlegend=False,
    margin=dict(r=160),
    xaxis_title="Year", yaxis_title="Price-to-Income Ratio",
)
fig3.show()
save_chart(fig3, 3, "lines")

## Chart 4 — Small multiples: price and income indexed to 1995 = 100

**Research question 1:** How have house prices and income evolved since 1995?

**What to look for:** Price lines that rise much faster than income lines —
the gap between them is the affordability squeeze.

In [ ]:
_sm = df[(df["property_type"] == PROPERTY_TYPE) & (df["family_type"] == FAMILY_TYPE)].copy()

# Index to 1995 = 100
_base = _sm[_sm["year"] == 1995][["landsdel", "avg_price_dkk", INCOME_COL]].rename(
    columns={"avg_price_dkk": "price_base", INCOME_COL: "income_base"}
)
_sm = _sm.merge(_base, on="landsdel")
_sm["price_index"]  = _sm["avg_price_dkk"] / _sm["price_base"] * 100
_sm["income_index"] = _sm[INCOME_COL] / _sm["income_base"] * 100

NROWS, NCOLS = 3, 4
fig4 = make_subplots(
    rows=NROWS, cols=NCOLS,
    subplot_titles=LANDSDEL_ORDER,
    shared_yaxes=True,
    horizontal_spacing=0.04, vertical_spacing=0.08,
)

PRICE_COLOR  = "#2a78d6"
INCOME_COLOR = "#eb6834"

for i, ld in enumerate(LANDSDEL_ORDER):
    r = i // NCOLS + 1
    c = i % NCOLS + 1
    sub = _sm[_sm["landsdel"] == ld].sort_values("year")
    fig4.add_trace(
        go.Scatter(x=sub["year"], y=sub["price_index"], mode="lines",
                   name="Price", legendgroup="price",
                   line=dict(color=PRICE_COLOR, width=2),
                   showlegend=(i == 0),
                   hovertemplate="%{y:.0f}<extra>Price index</extra>"),
        row=r, col=c,
    )
    fig4.add_trace(
        go.Scatter(x=sub["year"], y=sub["income_index"], mode="lines",
                   name="Income", legendgroup="income",
                   line=dict(color=INCOME_COLOR, width=2),
                   showlegend=(i == 0),
                   hovertemplate="%{y:.0f}<extra>Income index</extra>"),
        row=r, col=c,
    )

fig4.update_yaxes(title_text="Index (1995 = 100)", row=1, col=1)
fig4.update_yaxes(title_text="Index (1995 = 100)", row=2, col=1)
fig4.update_yaxes(title_text="Index (1995 = 100)", row=3, col=1)
fig4.update_layout(
    height=800, title_text="Price vs Income growth by region (1995 = 100)",
    legend=dict(orientation="h", yanchor="bottom", y=-0.08),
)
fig4.show()
save_chart(fig4, 4, "small-multiples", png_height=800)

## Chart 5 — Animated bubble scatter (log-log)

**Research question 3:** Is it worth living in Copenhagen? Is the affordability gap
growing?

**What to look for:** Bubbles drifting above the diagonal reference lines mean
worsening affordability. Copenhagen bubbles are small (≈1 000 house sales/year —
thin market), so they're harder to spot despite having the highest ratios.

Note: 11 series in an all-pairs chart (bubble) exceeds the colourblind-safe limit
of 3. Use small multiples or emphasis in the final dashboard.

In [ ]:
_bub = df[(df["property_type"] == PROPERTY_TYPE) & (df["family_type"] == FAMILY_TYPE)].copy()
_bub["sales_note"] = _bub["n_sales"].apply(low_sample_note)

# Fix axis ranges across all frames
_xmin = _bub[INCOME_COL].min() * 0.7
_xmax = _bub[INCOME_COL].max() * 1.3
_ymin = _bub["avg_price_dkk"].min() * 0.7
_ymax = _bub["avg_price_dkk"].max() * 1.3

fig5 = px.scatter(
    _bub.sort_values("year"),
    x=INCOME_COL, y="avg_price_dkk",
    size="n_sales", color="landsdel",
    animation_frame="year",
    color_discrete_map=LANDSDEL_COLORS,
    category_orders={"landsdel": LANDSDEL_ORDER},
    hover_name="landsdel",
    hover_data={INCOME_COL: ":,.0f", "avg_price_dkk": ":,.0f",
                "sales_note": True, "year": False, "n_sales": False, "landsdel": False},
    log_x=True, log_y=True,
    size_max=45,
    labels={INCOME_COL: f"Avg {INCOME_BASIS} income (DKK)", "avg_price_dkk": "Avg price (DKK)",
            "sales_note": "Sales"},
    title="Income vs Price (bubble size = number of sales)",
)

fig5.update_xaxes(range=[np.log10(_xmin), np.log10(_xmax)])
fig5.update_yaxes(range=[np.log10(_ymin), np.log10(_ymax)])

# Dashed reference lines for constant ratios (price = ratio × income)
_ref_x = np.array([_xmin, _xmax])
for ratio_val, label in [(3, "3×"), (5, "5×"), (10, "10×")]:
    _ref_y = _ref_x * ratio_val
    fig5.add_trace(go.Scatter(
        x=_ref_x, y=_ref_y, mode="lines",
        line=dict(dash="dash", color="rgba(128,128,128,0.4)", width=1),
        name=label, showlegend=False, hoverinfo="skip",
    ))
    fig5.add_annotation(
        x=np.log10(_xmax * 0.95), y=np.log10(ratio_val * _xmax * 0.95),
        text=label, showarrow=False, font=dict(size=10, color="gray"),
        xref="x", yref="y",
    )

fig5.update_layout(xaxis_tickformat=",", yaxis_tickformat=",")
fig5.show()

# Static PNG for the YEAR frame
_snap5 = _bub[_bub["year"] == YEAR]
fig5s = px.scatter(
    _snap5, x=INCOME_COL, y="avg_price_dkk",
    size="n_sales", color="landsdel",
    color_discrete_map=LANDSDEL_COLORS,
    category_orders={"landsdel": LANDSDEL_ORDER},
    hover_name="landsdel",
    log_x=True, log_y=True, size_max=45,
    labels={INCOME_COL: f"Avg {INCOME_BASIS} income (DKK)", "avg_price_dkk": "Avg price (DKK)"},
    title=f"Income vs Price, {YEAR}",
)
fig5s.update_xaxes(range=[np.log10(_xmin), np.log10(_xmax)])
fig5s.update_yaxes(range=[np.log10(_ymin), np.log10(_ymax)])
for ratio_val, label in [(3, "3×"), (5, "5×"), (10, "10×")]:
    _ref_y = _ref_x * ratio_val
    fig5s.add_trace(go.Scatter(
        x=_ref_x, y=_ref_y, mode="lines",
        line=dict(dash="dash", color="rgba(128,128,128,0.4)", width=1),
        showlegend=False, hoverinfo="skip",
    ))
    fig5s.add_annotation(
        x=np.log10(_xmax * 0.95), y=np.log10(ratio_val * _xmax * 0.95),
        text=label, showarrow=False, font=dict(size=10, color="gray"),
    )
fig5s.update_layout(xaxis_tickformat=",", yaxis_tickformat=",")

save_chart(fig5, 5, "bubble", png_to_report=False)
save_chart(fig5s, 5, "bubble")

## Chart 6 — Heatmap: landsdel × year

**Research questions 1 & 1a:** How have prices evolved? What is the trend per
province?

**What to look for:** Dark blocks = expensive regions/years. Rows sorted by the
2024 ratio (most expensive on top).

In [ ]:
_heat = df[(df["property_type"] == PROPERTY_TYPE) & (df["family_type"] == FAMILY_TYPE)].copy()

# Sort rows by 2024 ratio (most expensive on top)
_order_2024 = (
    _heat[_heat["year"] == 2024]
    .sort_values(RATIO_COL, ascending=True)["landsdel"]
    .tolist()
)

_pivot = _heat.pivot_table(index="landsdel", columns="year", values=RATIO_COL)
_pivot = _pivot.loc[_order_2024]

_pivot_sales = _heat.pivot_table(index="landsdel", columns="year", values="n_sales")
_pivot_sales = _pivot_sales.loc[_order_2024]

# Build custom hover text
_hovertext = []
for ld in _order_2024:
    row = []
    for yr in _pivot.columns:
        ratio = _pivot.loc[ld, yr]
        sales = _pivot_sales.loc[ld, yr]
        note = " ⚠ low sample" if sales < 15 else ""
        row.append(f"{ld}<br>Year: {yr}<br>Ratio: {ratio:.2f}<br>Sales: {sales:,.0f}{note}")
    _hovertext.append(row)

fig6 = go.Figure(go.Heatmap(
    z=_pivot.values,
    x=_pivot.columns.tolist(),
    y=_pivot.index.tolist(),
    colorscale=SEQ_BLUE,
    hovertext=_hovertext,
    hoverinfo="text",
    colorbar=dict(title="Ratio"),
))
fig6.update_layout(
    title=f"Affordability ratio heatmap ({INCOME_BASIS} income)",
    xaxis_title="Year", yaxis_title="",
    yaxis=dict(autorange="reversed"),
    height=500,
)
fig6.show()
save_chart(fig6, 6, "heatmap", png_height=500)

## Chart 7 — Box plot of regional ratios per year

**Research question 1:** How have prices and income evolved? How has the spread
between regions changed?

**What to look for:** Rising median = nationwide deterioration. A widening box =
growing inter-regional inequality.

In [ ]:
_box = df[(df["property_type"] == PROPERTY_TYPE) & (df["family_type"] == FAMILY_TYPE)].copy()
_box["year_str"] = _box["year"].astype(str)

fig7 = px.box(
    _box, x="year_str", y=RATIO_COL,
    hover_data={"landsdel": True, RATIO_COL: ":.2f", "n_sales": ":,", "year_str": False},
    labels={RATIO_COL: "Price-to-Income Ratio", "year_str": "Year",
            "landsdel": "Landsdel", "n_sales": "Sales"},
    title="Spread of regional affordability ratios per year",
)
fig7.update_traces(marker_color="#2a78d6", line_color="#2a78d6")
fig7.update_layout(xaxis_title="Year", yaxis_title="Price-to-Income Ratio")
fig7.show()
save_chart(fig7, 7, "boxplot")

## Chart 8 — Couples vs Singles (dumbbell)

**Research question 1b:** How has affordability changed for couples compared to
singles?

**What to look for:** The gap between dots = how much worse off singles are.
Note: singles' ratios are hypothetical — the same average sale price divided by a
single person's income, not prices of homes actually bought by singles.

In [ ]:
_c = df[(df["year"] == YEAR) & (df["property_type"] == PROPERTY_TYPE) & (df["family_type"] == "couples")][[
    "landsdel", RATIO_COL, "n_sales"
]].rename(columns={RATIO_COL: "ratio_couples", "n_sales": "n_sales_couples"})

_s = df[(df["year"] == YEAR) & (df["property_type"] == PROPERTY_TYPE) & (df["family_type"] == "singles")][[
    "landsdel", RATIO_COL, "n_sales"
]].rename(columns={RATIO_COL: "ratio_singles", "n_sales": "n_sales_singles"})

_db = _c.merge(_s, on="landsdel").sort_values("ratio_singles", ascending=True)
_ld_sorted = _db["landsdel"].tolist()

COUPLES_COLOR = "#2a78d6"
SINGLES_COLOR = "#e87ba4"

fig8 = go.Figure()

# Connecting lines
for _, r in _db.iterrows():
    fig8.add_trace(go.Scatter(
        x=[r["ratio_couples"], r["ratio_singles"]],
        y=[r["landsdel"], r["landsdel"]],
        mode="lines", line=dict(color="#c3c2b7", width=2),
        showlegend=False, hoverinfo="skip",
    ))

# Couples dots
fig8.add_trace(go.Scatter(
    x=_db["ratio_couples"], y=_db["landsdel"], mode="markers",
    marker=dict(size=10, color=COUPLES_COLOR),
    name="Couples",
    customdata=_db["n_sales_couples"],
    hovertemplate="Couples: %{x:.2f}<br>Sales: %{customdata:,}<extra>%{y}</extra>",
))

# Singles dots
fig8.add_trace(go.Scatter(
    x=_db["ratio_singles"], y=_db["landsdel"], mode="markers",
    marker=dict(size=10, color=SINGLES_COLOR),
    name="Singles (hypothetical)",
    customdata=_db["n_sales_singles"],
    hovertemplate="Singles: %{x:.2f}<br>Sales: %{customdata:,}<extra>%{y}</extra>",
))

fig8.update_layout(
    title=f"Couples vs Singles affordability, {YEAR}<br>"
          f"<sup>Singles' ratio is hypothetical: same avg price ÷ single income</sup>",
    xaxis_title="Price-to-Income Ratio",
    yaxis=dict(categoryorder="array", categoryarray=_ld_sorted),
    legend=dict(orientation="h", yanchor="bottom", y=-0.15),
    height=550,
)
fig8.show()
save_chart(fig8, 8, "dumbbell-family", png_height=550)

## Chart 9 — House vs Flat per region (dumbbell)

**Research question 3:** Is it worth living in Copenhagen? Is the gap between
Copenhagen and other provinces growing?

**What to look for:** Regions where flats are more expensive relative to income
than houses (or vice versa). In Byen København, houses are a thin market
(≈1 000 sales/year), so the house ratio there should be read with caution.

In [ ]:
_h = df[(df["year"] == YEAR) & (df["property_type"] == "one_family_house") & (df["family_type"] == FAMILY_TYPE)][[
    "landsdel", RATIO_COL, "n_sales"
]].rename(columns={RATIO_COL: "ratio_house", "n_sales": "sales_house"})

_f = df[(df["year"] == YEAR) & (df["property_type"] == "owner_occupied_flat") & (df["family_type"] == FAMILY_TYPE)][[
    "landsdel", RATIO_COL, "n_sales"
]].rename(columns={RATIO_COL: "ratio_flat", "n_sales": "sales_flat"})

_hf = _h.merge(_f, on="landsdel")
_hf["max_ratio"] = _hf[["ratio_house", "ratio_flat"]].max(axis=1)
_hf = _hf.sort_values("max_ratio", ascending=True)
_ld_sorted9 = _hf["landsdel"].tolist()

HOUSE_COLOR = "#2a78d6"
FLAT_COLOR  = "#eb6834"

fig9 = go.Figure()

for _, r in _hf.iterrows():
    fig9.add_trace(go.Scatter(
        x=[r["ratio_house"], r["ratio_flat"]],
        y=[r["landsdel"], r["landsdel"]],
        mode="lines", line=dict(color="#c3c2b7", width=2),
        showlegend=False, hoverinfo="skip",
    ))

fig9.add_trace(go.Scatter(
    x=_hf["ratio_house"], y=_hf["landsdel"], mode="markers",
    marker=dict(size=10, color=HOUSE_COLOR), name="House",
    customdata=np.stack([_hf["sales_house"], _hf["sales_house"].apply(
        lambda v: " ⚠ low" if v < 15 else "")], axis=-1),
    hovertemplate="House: %{x:.2f}<br>Sales: %{customdata[0]:,}%{customdata[1]}<extra>%{y}</extra>",
))

fig9.add_trace(go.Scatter(
    x=_hf["ratio_flat"], y=_hf["landsdel"], mode="markers",
    marker=dict(size=10, color=FLAT_COLOR), name="Flat",
    customdata=np.stack([_hf["sales_flat"], _hf["sales_flat"].apply(
        lambda v: " ⚠ low" if v < 15 else "")], axis=-1),
    hovertemplate="Flat: %{x:.2f}<br>Sales: %{customdata[0]:,}%{customdata[1]}<extra>%{y}</extra>",
))

fig9.update_layout(
    title=f"House vs Flat affordability, {YEAR}",
    xaxis_title="Price-to-Income Ratio",
    yaxis=dict(categoryorder="array", categoryarray=_ld_sorted9),
    legend=dict(orientation="h", yanchor="bottom", y=-0.15),
    height=550,
)
fig9.show()
save_chart(fig9, 9, "dumbbell-property", png_height=550)

## Chart 10 — East vs West over time (weighted)

**Research question 2:** Is housing more affordable east or west of the Great Belt?

**What to look for:** Persistent gap between the two lines; whether it widens or
narrows over time.

This is a reference chart — we plan to produce the AI-generated version of this
graph separately, so we can compare. HTML only, no PNG export.

In [ ]:
_ew = df[(df["property_type"] == PROPERTY_TYPE) & (df["family_type"] == FAMILY_TYPE)].copy()
_agg = _ew.groupby(["year", "side"]).apply(weighted_aggregate, include_groups=False).reset_index()

fig10 = px.line(
    _agg, x="year", y=RATIO_COL, color="side",
    color_discrete_map=SIDE_COLORS,
    hover_data={RATIO_COL: ":.2f", "n_sales": ":,", "side": False},
    labels={RATIO_COL: "Weighted Price-to-Income Ratio", "year": "Year",
            "side": "Great Belt side", "n_sales": "Total sales"},
    title="Weighted affordability: east vs west of the Great Belt",
)

# Direct labels
for side in ["east", "west"]:
    row = _agg[(_agg["year"] == _agg["year"].max()) & (_agg["side"] == side)].iloc[0]
    fig10.add_annotation(
        x=row["year"], y=row[RATIO_COL],
        text=side.title(), showarrow=False, xanchor="left", xshift=6,
        font=dict(size=11, color=SIDE_COLORS[side]),
    )

fig10.update_layout(showlegend=False, margin=dict(r=80))
fig10.show()
save_chart(fig10, 10, "east-west", png_to_report=False)

## Summary

All 10 charts generated. HTML versions are in `notebooks/figures/`.
PNG versions for charts 1–9 are in `report/figures/` as `g1-*.png` to `g9-*.png`.

**Known limitations for the final dashboard:**
- Charts 3 and 5 use 11 categorical colours, exceeding the colourblind-safe limit
  of 8. Use emphasis or small multiples instead.
- Chart 1 (choropleth): Bornholm inset not yet implemented; it's a tiny island
  that may be hard to see.
- Chart 5 (bubble): reference-line annotations may overlap on some axis ranges.